# CineIQ — Content-Based Filtering

Uses genome tag relevance scores + genre features with top-K sparse similarity.

**Why top-K sparse?**
- 27K movies × 27K dense matrix = ~5GB. Doesn't fit in memory.
- Top-K stores only top-50 similar movies per movie.
- Memory: ~10MB. Query time: O(1) dictionary lookup.
- Production-grade approach used by Netflix, YouTube.

In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import time
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors

RAW = "../data/raw/ml-20m"
PROC = "../data/processed"
MODELS = "../models"

movies = pd.read_csv(f"{RAW}/movies.csv")
print(f"Movies: {len(movies):,}")

Movies: 27,278


## 1. Load Genome Scores

Genome scores provide 1128 tag relevance features per movie (e.g., 'based on a book', 'aliens', 'dark comedy').

In [2]:
# Load genome scores (11.7M rows)
t0 = time.time()
genome = pd.read_csv(f"{RAW}/genome-scores.csv")
print(f"Loaded genome scores: {len(genome):,} rows in {time.time() - t0:.1f}s")
print(f"Columns: {genome.columns.tolist()}")
print(f"Unique movies: {genome['movieId'].nunique():,}")
print(f"Unique tags: {genome['tagId'].nunique():,}")
genome.head()

Loaded genome scores: 11,709,768 rows in 2.2s
Columns: ['movieId', 'tagId', 'relevance']
Unique movies: 10,381
Unique tags: 1,128


,movieId,tagId,relevance
0,1,1,0.02500
1,1,2,0.02500
2,1,3,0.05775
3,1,4,0.09675
4,1,5,0.14675


In [3]:
# Load tag names for reference
tags = pd.read_csv(f"{RAW}/genome-tags.csv")
print(f"Tags: {len(tags)}")
tags.head(10)

Tags: 1128


,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s
5,6,1950s
6,7,1960s
7,8,1970s
8,9,1980s
9,10,19th century


## 2. Build Feature Matrix

Pivot genome scores into a movie × tag matrix, then combine with genre features.

In [4]:
# Pivot genome scores: movies x tags matrix (27K x 1128)
t0 = time.time()
genome_matrix = genome.pivot(index="movieId", columns="tagId", values="relevance")
genome_matrix = genome_matrix.fillna(0)
print(f"Genome matrix shape: {genome_matrix.shape}")
print(f"Built in {time.time() - t0:.1f}s")
genome_matrix.head()

Genome matrix shape: (10381, 1128)
Built in 3.8s


tagId,1,2,3,4,5,6,7,8,9,10,...,1119,1120,1121,1122,1123,1124,1125,1126,1127,1128
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.02500,0.02500,0.05775,0.09675,0.14675,0.21700,0.06700,0.26275,0.26200,0.03200,...,0.03950,0.01800,0.04575,0.03275,0.12500,0.04150,0.01925,0.03625,0.07775,0.02300
2,0.03975,0.04375,0.03775,0.04800,0.11025,0.07250,0.04775,0.10975,0.09925,0.02050,...,0.04175,0.01925,0.01725,0.02425,0.12550,0.02250,0.01550,0.01475,0.09025,0.01875
3,0.04350,0.05475,0.02800,0.07700,0.05400,0.06850,0.05600,0.18500,0.04925,0.02675,...,0.04150,0.02675,0.02775,0.03425,0.15550,0.03675,0.01700,0.01950,0.09700,0.01850
4,0.03725,0.03950,0.03675,0.03100,0.06825,0.04050,0.02325,0.08700,0.05125,0.03025,...,0.05750,0.03375,0.02275,0.03975,0.18525,0.05925,0.01500,0.01525,0.06450,0.01300
5,0.04200,0.05275,0.05925,0.03675,0.07525,0.12525,0.02850,0.08500,0.02950,0.02875,...,0.04250,0.02825,0.02150,0.02600,0.14275,0.02075,0.01650,0.01675,0.10750,0.01825


In [5]:
# Add genre features as binary columns
movies["genre_list"] = movies["genres"].apply(lambda x: x.split("|") if x != "(no genres listed)" else [])
all_genres = sorted(set(g for glist in movies["genre_list"] for g in glist))
print(f"Unique genres: {len(all_genres)}")

genre_df = pd.DataFrame(0, index=movies["movieId"], columns=all_genres)
for _, row in movies.iterrows():
    if row["movieId"] in genre_df.index:
        for g in row["genre_list"]:
            genre_df.loc[row["movieId"], g] = 1.0

print(f"Genre matrix shape: {genre_df.shape}")

Unique genres: 19
Genre matrix shape: (27278, 19)


In [6]:
# Combine genome + genre features (weight genres more since they're stronger signal)
GENOME_WEIGHT = 1.0
GENRE_WEIGHT = 3.0

# Align on common movieIds
common_ids = sorted(set(genome_matrix.index) & set(genre_df.index))
print(f"Common movies: {len(common_ids):,}")

genome_aligned = genome_matrix.loc[common_ids]
genre_aligned = genre_df.loc[common_ids]

# Weight and concatenate
combined = pd.concat([
    genome_aligned * GENOME_WEIGHT,
    genre_aligned * GENRE_WEIGHT
], axis=1)

print(f"Combined feature matrix: {combined.shape}")
print(f"Total features: {combined.shape[1]} (1128 genome + {len(all_genres)} genres)")

Common movies: 10,381
Combined feature matrix: (10381, 1147)
Total features: 1147 (1128 genome + 19 genres)


In [7]:
# Normalize rows to unit vectors (for cosine similarity via dot product)
combined_normalized = normalize(combined.values, norm="l2")
print(f"Normalized shape: {combined_normalized.shape}")

Normalized shape: (10381, 1147)


## 3. Build Top-K Similarity Index

For each movie, find top-50 most similar movies using brute-force nearest neighbors.
Store only the top-K neighbors per movie → ~10MB.

In [8]:
# Build NearestNeighbors index
t0 = time.time()
K = 50

nn = NearestNeighbors(n_neighbors=K + 1, metric="cosine", algorithm="brute")
nn.fit(combined_normalized)
print(f"Index built in {time.time() - t0:.1f}s")

# Query for all movies at once
distances, indices = nn.kneighbors(combined_normalized)
print(f"Distances shape: {distances.shape}")
print(f"Built in {time.time() - t0:.1f}s total")

Index built in 0.1s
Distances shape: (10381, 51)
Built in 8.6s total


In [9]:
# Build sparse index: {movieId: [(similarId, score), ...]}
movie_ids = common_ids
content_index = {}

for i, movie_id in enumerate(movie_ids):
    neighbors = []
    for j in range(1, K + 1):  # skip self (index 0)
        neighbor_id = int(movie_ids[indices[i, j]])
        similarity = float(1.0 - distances[i, j])  # cosine distance -> similarity
        neighbors.append([neighbor_id, round(similarity, 6)])
    content_index[str(movie_id)] = neighbors

print(f"Content index: {len(content_index):,} movies")
print(f"Sample (Toy Story):")
# Find Toy Story
toy_story_id = movies[movies["title"] == "Toy Story (1995)"]["movieId"].values[0]
if str(toy_story_id) in content_index:
    for nid, score in content_index[str(toy_story_id)][:5]:
        title = movies[movies["movieId"] == nid]["title"].values
        if len(title) > 0:
            print(f"  {title[0]}: {score:.4f}")

Content index: 10,381 movies
Sample (Toy Story):
  Monsters, Inc. (2001): 0.9745
  Toy Story 2 (1999): 0.9726
  Antz (1998): 0.9353
  Bug's Life, A (1998): 0.9283
  Emperor's New Groove, The (2000): 0.9177


In [10]:
# Check memory size
import sys
index_size = sys.getsizeof(content_index)
# More accurate: check actual serialized size
import io
buffer = io.BytesIO()
pickle.dump(content_index, buffer)
actual_size = buffer.tell()
print(f"Content index size: {actual_size / 1024 / 1024:.1f} MB")
print(f"vs dense matrix would be: {len(common_ids) * len(common_ids) * 8 / 1024 / 1024 / 1024:.1f} GB")

Content index size: 8.2 MB
vs dense matrix would be: 0.8 GB


## 4. Save Artifacts

In [11]:
# Save content index as JSON (compact, human-readable)
with open(f"{MODELS}/content_index.json", "w") as f:
    json.dump(content_index, f)
print(f"Saved content_index.json ({actual_size / 1024 / 1024:.1f} MB)")

# Also save movieId mapping for the API
title_to_id = dict(zip(movies["title"], movies["movieId"]))
id_to_title = dict(zip(movies["movieId"], movies["title"]))
with open(f"{MODELS}/movie mappings.json", "w") as f:
    json.dump({"title_to_id": {k: int(v) for k, v in title_to_id.items()},
               "id_to_title": {str(k): v for k, v in id_to_title.items()}}, f)
print("Saved movie_mappings.json")

Saved content_index.json (8.2 MB)
Saved movie_mappings.json


## 5. Verify

In [12]:
# Test: similar movies to The Matrix
# Note: 20M dataset uses 'Title, The (Year)' format
matrix_id = 2571  # Matrix, The (1999)
print(f'Matrix, The (1999) — movieId: {matrix_id}')
print(f'Top 10 similar movies:')
for nid, score in content_index[str(matrix_id)][:10]:
    title = movies[movies['movieId'] == nid]['title'].values
    if len(title) > 0:
        print(f'  {title[0]}: {score:.4f}')

Matrix, The (1999) — movieId: 2571
Top 10 similar movies:
  Equilibrium (2002): 0.9193
  Terminator, The (1984): 0.8951
  Blade Runner (1982): 0.8950
  Terminator 2: Judgment Day (1991): 0.8641
  Total Recall (1990): 0.8626
  Colossus: The Forbin Project (1970): 0.8616
  Abyss, The (1989): 0.8549
  Matrix Reloaded, The (2003): 0.8481
  Edge of Tomorrow (2014): 0.8458
  I, Robot (2004): 0.8452
